# Lab 10 — Métodos de Propagación de Etiquetas
**Grupo 03** | Minería de Datos — Sección 20  
Universidad del Valle de Guatemala

**Integrantes:** Felipe Aguilar, Vianka Castro, Nicolás Concuá, Ricardo Godínez, Fernando Hernández, Fernando Rueda

---

## Fase 2 — Selección y Análisis de Dataset

**Dataset:** Red Wine Quality  
**Fuente:** UCI Machine Learning Repository / Kaggle  
**Algoritmos a aplicar:** Label Propagation & Label Spreading

---
## 2a. Importación del Dataset

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('winequality-red.csv', sep=';')

print('Dataset cargado exitosamente.')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

In [ ]:
df.head()

In [ ]:
print('Columnas del dataset:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
feature_cols = [c for c in df.columns if c != 'quality']
X = df[feature_cols]
y_raw = df['quality']

df['quality_label'] = (df['quality'] >= 7).astype(int)
df['quality_str']   = df['quality_label'].map({0: 'bad', 1: 'good'})
y = df['quality_label']

print(f'Features : {len(feature_cols)}')
print(f'Target   : quality binarizado (>=7 = good, <7 = bad)')
print(f'Muestras : {len(df)}')

---
## 2b. Análisis Exploratorio de Datos (EDA)

### Tipos de variables y dimensionalidad

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

In [ ]:
type_summary = pd.DataFrame({
    'dtype'      : df[feature_cols].dtypes,
    'non_null'   : df[feature_cols].notna().sum(),
    'null'       : df[feature_cols].isna().sum(),
    'unique_vals': df[feature_cols].nunique()
})

print('=== Resumen de tipos de variables ===')
print(type_summary.to_string())
print(f'\nFeatures numericas  : {df[feature_cols].select_dtypes(include=np.number).shape[1]}')
print(f'Features categoricas: {df[feature_cols].select_dtypes(include="object").shape[1]}')

In [ ]:
print(f'Dimensionalidad del espacio de features: {X.shape[1]} dimensiones')
print(f'Numero de muestras                     : {X.shape[0]}')
print(f'\nDistribucion de scores de calidad originales:')
print(y_raw.value_counts().sort_index().to_string())

### Balance de clases

In [ ]:
class_counts = df['quality_str'].value_counts()
class_pct    = df['quality_str'].value_counts(normalize=True) * 100

print('=== Balance de clases (binarizado) ===')
for cls in class_counts.index:
    print(f'  {cls:6s}: {class_counts[cls]:5d} muestras ({class_pct[cls]:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución score original
score_counts = y_raw.value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values, color='#8e44ad', edgecolor='white', linewidth=0.8)
axes[0].axvline(x=6.5, color='red', linestyle='--', linewidth=1.5, label='umbral binarizacion (7)')
axes[0].set_title('Distribucion scores originales')
axes[0].set_xlabel('Quality score')
axes[0].set_ylabel('Cantidad')
axes[0].legend(fontsize=8)
for i, (k, v) in enumerate(score_counts.items()):
    axes[0].text(k, v + 5, str(v), ha='center', fontsize=9)

# Countplot binarizado
palette = {'bad': '#e74c3c', 'good': '#2ecc71'}
sns.countplot(data=df, x='quality_str', ax=axes[1],
              order=['bad', 'good'], palette=palette)
axes[1].set_title('Distribucion clases (binarizado)')
axes[1].set_xlabel('Calidad')
axes[1].set_ylabel('Cantidad')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
axes[2].pie(class_counts.values,
            labels=class_counts.index,
            autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'],
            startangle=90,
            explode=(0, 0.05))
axes[2].set_title('Proporcion de clases')

plt.suptitle('Balance de clases — Red Wine Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Estadísticas descriptivas

In [ ]:
desc = X.describe().T
desc['cv'] = (desc['std'] / desc['mean']).round(3)
desc = desc[['mean', 'std', 'cv', 'min', '25%', '50%', '75%', 'max']]
desc.columns = ['Media', 'Std', 'CV', 'Min', 'Q1', 'Mediana', 'Q3', 'Max']

print('=== Estadisticas descriptivas (features) ===')
desc.round(4)

### Distribución de features por clase

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    sns.histplot(data=df, x=feat, hue='quality_str',
                 ax=axes[i], kde=True,
                 palette=palette, alpha=0.55, bins=30)
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')
    if i != 0:
        axes[i].get_legend().remove()

# Ocultar subplot sobrante
axes[-1].set_visible(False)

plt.suptitle('Distribucion de features por clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Boxplots por clase

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    sns.boxplot(data=df, x='quality_str', y=feat,
                ax=axes[i], palette=palette,
                order=['bad', 'good'])
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)

plt.suptitle('Boxplots por clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Matriz de correlación

In [ ]:
corr_matrix = X.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title('Matriz de correlacion — features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

print('Pares con |r| > 0.70:')
for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {a:30s} <-> {b:30s}  r = {r}')

### Correlación de cada feature con el score de calidad

In [ ]:
corr_with_target = X.corrwith(y_raw).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlacion de Pearson con quality')
ax.set_title('Correlacion de features con el score de calidad', fontsize=12, fontweight='bold')
for i, (feat, val) in enumerate(corr_with_target.items()):
    ax.text(val + (0.005 if val >= 0 else -0.005), i,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

### Detección de outliers (IQR)

In [ ]:
Q1  = X.quantile(0.25)
Q3  = X.quantile(0.75)
IQR = Q3 - Q1

outlier_mask   = (X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))
outlier_counts = outlier_mask.sum().sort_values(ascending=False)

print('=== Outliers por feature (criterio IQR) ===')
print(outlier_counts[outlier_counts > 0].to_string())
print(f'\nMuestras con al menos 1 outlier: {outlier_mask.any(axis=1).sum()} / {len(df)}')

top5 = outlier_counts.head(5).index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, feat in enumerate(top5):
    sns.boxplot(data=df, y=feat, x='quality_str',
                ax=axes[i], palette=palette, order=['bad', 'good'])
    axes[i].set_title(feat[:20], fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Top 5 features con mas outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Proyección PCA — separabilidad visual

In [ ]:
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

colors = y.map({0: '#e74c3c', 1: '#2ecc71'})
legend_elements = [Patch(facecolor='#e74c3c', label='bad  (<7)'),
                   Patch(facecolor='#2ecc71', label='good (>=7)')]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.5,
           edgecolors='k', linewidths=0.2, s=30)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Proyeccion PCA — Red Wine Quality', fontsize=12, fontweight='bold')
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print(f'Varianza explicada acumulada: {sum(pca.explained_variance_ratio_)*100:.2f}%')

### Valores faltantes

In [ ]:
missing = X.isna().sum()
print('=== Valores faltantes por feature ===')
if missing.sum() == 0:
    print('  No se encontraron valores faltantes. OK')
else:
    print(missing[missing > 0])